In [10]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import shap
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/feature_matrix.csv')
print(df.shape)
print(df.head())

(349, 10)
       Disease  Fever  Cough  Fatigue  Difficulty Breathing  Age  Gender  \
0    Influenza      1      0        1                     1   19       0   
1  Common Cold      0      1        1                     0   25       0   
2       Eczema      0      1        1                     0   25       0   
3       Asthma      1      1        0                     1   25       1   
4       Asthma      1      1        0                     1   25       1   

   Blood Pressure  Cholesterol Level  Outcome Variable  
0               0                  1                 1  
1               1                  1                 0  
2               1                  1                 0  
3               1                  1                 1  
4               1                  1                 1  


In [5]:
# Quick audit of positive/negative rows per disease
summary = df.groupby('Disease')['Outcome Variable'].agg(
    total='count',
    positive='sum',
    negative=lambda x: (x == 0).sum()
).reset_index()

print('=== Dataset Audit ===')
print(f'Total diseases: {df["Disease"].nunique()}')
print(f'Diseases with at least 1 positive row: {(summary["positive"] > 0).sum()}')
print(f'Diseases with zero positive rows: {(summary["positive"] == 0).sum()}')
print(f'Total rows: {len(df)}')
print(f'Total positive rows: {(df["Outcome Variable"] == 1).sum()}')
print(f'Total negative rows: {(df["Outcome Variable"] == 0).sum()}')
print()
print('Diseases with NO positive rows (cannot be predicted):')
no_positive = summary[summary['positive'] == 0]['Disease'].tolist()
for d in no_positive:
    print(f'  - {d}')

=== Dataset Audit ===
Total diseases: 116
Diseases with at least 1 positive row: 77
Diseases with zero positive rows: 39
Total rows: 349
Total positive rows: 186
Total negative rows: 163

Diseases with NO positive rows (cannot be predicted):
  - Acne
  - Anemia
  - Appendicitis
  - Atherosclerosis
  - Autism Spectrum Disorder (ASD)
  - Bladder Cancer
  - Brain Tumor
  - Breast Cancer
  - Cholecystitis
  - Cholera
  - Chronic Obstructive Pulmonary...
  - Cystic Fibrosis
  - Dengue Fever
  - Endometriosis
  - Epilepsy
  - Esophageal Cancer
  - Glaucoma
  - HIV/AIDS
  - Hepatitis
  - Hyperglycemia
  - Marfan Syndrome
  - Measles
  - Melanoma
  - Muscular Dystrophy
  - Obsessive-Compulsive Disorde...
  - Osteomyelitis
  - Otitis Media (Ear Infection)
  - Ovarian Cancer
  - Polio
  - Prostate Cancer
  - Rabies
  - Rubella
  - Schizophrenia
  - Sepsis
  - Sinusitis
  - Sleep Apnea
  - Tourette Syndrome
  - Turner Syndrome
  - Zika Virus


Problem: over half of the diseases only have 1 row, so we can't do a train/test split. Further, 39 of the diseases have no positive rows, so a model can't predict what the disease looks like. Because of these limitations, we will use one-vs-rest binary classification, building labels that indicate confirmed results strictly when there is a positive case of the disease we are currently training.

In [6]:
feature_cols = [
    'Fever',
    'Cough',
    'Fatigue',
    'Difficulty Breathing',
    'Age',
    'Gender',
    'Blood Pressure',
    'Cholesterol Level'
]

# X does NOT include Outcome Variable or Disease
# Outcome Variable is used only for labelling, not as a feature
X = df[feature_cols]

print('Feature columns:', feature_cols)
print('Feature matrix shape:', X.shape)

Feature columns: ['Fever', 'Cough', 'Fatigue', 'Difficulty Breathing', 'Age', 'Gender', 'Blood Pressure', 'Cholesterol Level']
Feature matrix shape: (349, 8)


In [11]:
os.makedirs('../models/disease_models', exist_ok=True)

disease_models = {}
training_report = []

all_diseases = df['Disease'].unique()

for disease in all_diseases:
    # Label = 1 only when it IS this disease AND confirmed positive
    labels = ((df['Disease'] == disease) & (df['Outcome Variable'] == 1)).astype(int)

    positive_count = labels.sum()
    negative_count = (labels == 0).sum()

    # Skip diseases with no positive examples — nothing to learn
    if positive_count == 0:
        training_report.append({
            'disease': disease,
            'status': 'skipped',
            'positive_rows': 0,
            'negative_rows': negative_count
        })
        continue

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,       # let trees grow fully for small datasets
        min_samples_leaf=1,
        class_weight='balanced',  # critical — most rows are 0, this rebalances
        random_state=42
    )
    model.fit(X, labels)

    # Save the model
    safe_name = disease.replace(' ', '_').replace('/', '_').replace("'", '')
    joblib.dump(model, f'../models/disease_models/{safe_name}.pkl')

    disease_models[disease] = model
    training_report.append({
        'disease': disease,
        'status': 'trained',
        'positive_rows': int(positive_count),
        'negative_rows': int(negative_count)
    })

report_df = pd.DataFrame(training_report)
print(f'Successfully trained: {len(disease_models)} models')
print(f'Skipped (no positive rows): {(report_df["status"] == "skipped").sum()}')
print()
print('Trained diseases:')
print(report_df[report_df['status'] == 'trained'][['disease', 'positive_rows', 'negative_rows']])

Successfully trained: 77 models
Skipped (no positive rows): 39

Trained diseases:
               disease  positive_rows  negative_rows
0            Influenza              6            343
1          Common Cold              1            348
2               Eczema              3            346
3               Asthma             18            331
4      Hyperthyroidism              3            346
..                 ...            ...            ...
108              Mumps              2            347
112               Gout              1            348
113  Testicular Cancer              1            348
114        Tonsillitis              1            348
115  Williams Syndrome              1            348

[77 rows x 3 columns]


In [12]:
# Save feature column order — critical for consistent predictions in the app
joblib.dump(feature_cols, '../models/feature_columns.pkl')

# Save list of trained diseases — the app will only show these as possible outputs
trained_diseases = list(disease_models.keys())
joblib.dump(trained_diseases, '../models/trained_diseases.pkl')

# Save the full training report for reference
report_df.to_csv('../data/training_report.csv', index=False)

print(f'Saved {len(trained_diseases)} disease models')
print('Feature columns saved')
print('Training report saved to data/training_report.csv')

Saved 77 disease models
Feature columns saved
Training report saved to data/training_report.csv


In [13]:
def predict_diseases(user_features: dict, top_n=5):
    """
    Takes a dictionary of user inputs (no Outcome Variable needed),
    runs all disease models, and returns the top N predictions
    ranked by confidence.

    Parameters:
        user_features: dict with keys matching feature_cols
        top_n: how many predictions to return

    Returns:
        list of (disease_name, confidence_percentage) tuples
    """
    input_df = pd.DataFrame([user_features])[feature_cols]

    scores = {}
    for disease, model in disease_models.items():
        # predict_proba returns [prob_class_0, prob_class_1]
        # we want the probability of class 1 (disease confirmed)
        prob_positive = model.predict_proba(input_df)[0][1]
        scores[disease] = prob_positive

    # Sort by probability descending and take top N
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [(disease, round(prob * 100, 1)) for disease, prob in ranked[:top_n]]

Prediction function tests:

In [14]:
# Test 1 — classic flu symptoms
print('=== Test 1: Flu-like symptoms ===')
sample1 = {
    'Fever': 1,
    'Cough': 1,
    'Fatigue': 1,
    'Difficulty Breathing': 0,
    'Age': 35,
    'Gender': 1,
    'Blood Pressure': 1,
    'Cholesterol Level': 1
}
for disease, confidence in predict_diseases(sample1):
    print(f'  {disease}: {confidence}%')

print()

# Test 2 — respiratory symptoms, older patient, high blood pressure
print('=== Test 2: Respiratory + older patient ===')
sample2 = {
    'Fever': 0,
    'Cough': 1,
    'Fatigue': 1,
    'Difficulty Breathing': 1,
    'Age': 65,
    'Gender': 0,
    'Blood Pressure': 2,
    'Cholesterol Level': 2
}
for disease, confidence in predict_diseases(sample2):
    print(f'  {disease}: {confidence}%')

print()

# Test 3 — minimal symptoms, young patient
print('=== Test 3: Minimal symptoms, young patient ===')
sample3 = {
    'Fever': 0,
    'Cough': 0,
    'Fatigue': 1,
    'Difficulty Breathing': 0,
    'Age': 22,
    'Gender': 1,
    'Blood Pressure': 1,
    'Cholesterol Level': 0
}
for disease, confidence in predict_diseases(sample3):
    print(f'  {disease}: {confidence}%')

=== Test 1: Flu-like symptoms ===
  Rheumatoid Arthritis: 7.0%
  Asthma: 3.0%
  Influenza: 1.5%
  Hyperthyroidism: 1.0%
  Pancreatitis: 1.0%

=== Test 2: Respiratory + older patient ===
  Osteoporosis: 25.5%
  Stroke: 18.5%
  Liver Cancer: 16.3%
  Chronic Obstructive Pulmonary Disease (COPD): 15.5%
  Kidney Disease: 14.0%

=== Test 3: Minimal symptoms, young patient ===
  Rheumatoid Arthritis: 24.4%
  Influenza: 5.5%
  Colorectal Cancer: 2.5%
  Depression: 2.0%
  Cerebral Palsy: 1.5%


Different inputs produce different results, which is indicative of a working model. Obvious pitfalls include lack of symptoms/patient profile statistics to predict diseases from, leading to inaccurate results (many diseases that have the exact same symptoms and patient profiles can turn positive). Accuracy will improve if/when more data is incorporated into the model.